# 기계학습기반 단어중의성 해소

-  NLTK의 고전 데이터셋 Senseval(‘interest’, ‘hard’, ‘serve’) 를 이용해, 주변 문맥을 TF-IDF로 벡터화하고 로지스틱 회귀로 의미 분류

### 핵심 흐름
- 의미 태깅 코퍼스
- → 문맥 텍스트 추출
- → 벡터화
- → 지도학습 분류
- → 평가/예측

In [11]:
# ============================================
# WSD (Machine Learning-based) – Senseval + scikit-learn 데모 (Fixed)
# ============================================

# 1) 라이브러리 불러오기
import nltk
from nltk.corpus import senseval
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report
import numpy as np

In [12]:
# 2) 필요한 리소스 다운로드 (처음 한 번만)
nltk.download('senseval')
nltk.download('punkt')

# 3) 데이터셋 불러오기: 'interest' 의미 태깅 코퍼스
instances = senseval.instances('interest.pos')

[nltk_data] Downloading package senseval to /root/nltk_data...
[nltk_data]   Package senseval is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [13]:
def context_to_text(inst):
    """
    Senseval 인스턴스의 context에서 '단어'만 뽑아 공백으로 이어붙임.
    - context에는 문자열 또는 (단어, 품사) 튜플이 섞여 있음 → 튜플이면 첫 원소(단어) 사용
    - 공백/기호 제거, 소문자화
    """
    toks = []
    for w in inst.context:
        if isinstance(w, tuple):
            w = w[0]                 # (word, tag) → word
        if isinstance(w, str):
            w = w.strip().lower()
            if w and any(ch.isalpha() for ch in w):  # 최소 한 글자 이상 알파벳 포함
                toks.append(w)
    return " ".join(toks)

# 4) 입력(X)과 정답(y) 구성 + 빈 문서 제거
X_texts, y_labels = [], []
for inst in instances:
    text = context_to_text(inst)
    if text:                         # 빈 문서 제거
        X_texts.append(text)
        y_labels.append(inst.senses[0])

print(f"총 샘플 수(빈 문서 제거 후): {len(X_texts)}")
print(f"레이블 종류: {sorted(set(y_labels))}")
print("="*60)

총 샘플 수(빈 문서 제거 후): 2368
레이블 종류: ['interest_1', 'interest_2', 'interest_3', 'interest_4', 'interest_5', 'interest_6']


In [14]:
# 5) 학습/검증 분할
X_train, X_test, y_train, y_test = train_test_split(
    X_texts, y_labels, test_size=0.2, random_state=42, stratify=y_labels
)

# 6) 파이프라인 구성(빈 vocabulary 방지용 완화 설정)
pipeline = Pipeline([
    # token_pattern: 한 글자 토큰('i')도 허용, 숫자/밑줄 제외하고 알파벳만
    ("tfidf", TfidfVectorizer(
        ngram_range=(1, 2),
        min_df=1,                 # 문서 1개만 등장해도 허용
        max_df=0.95,              # 너무 흔한 토큰 일부만 제거
        token_pattern=r'(?u)\b[a-zA-Z]+\b'
    )),
    ("clf", LogisticRegression(max_iter=1000, n_jobs=None))
])

# 7) 학습
pipeline.fit(X_train, y_train)

Pipeline(steps=[('tfidf',
                 TfidfVectorizer(max_df=0.95, ngram_range=(1, 2),
                                 token_pattern='(?u)\\b[a-zA-Z]+\\b')),
                ('clf', LogisticRegression(max_iter=1000))])

In [15]:
# 8) 평가
pred = pipeline.predict(X_test)
acc = accuracy_score(y_test, pred)

print("📊 [모델 평가 결과]")
print(f"Accuracy: {acc:.3f}")
print("-"*60)
print("세부 성능 보고서:")
print(classification_report(y_test, pred, zero_division=0))
print("="*60)

📊 [모델 평가 결과]
Accuracy: 0.804
------------------------------------------------------------
세부 성능 보고서:
              precision    recall  f1-score   support

  interest_1       0.92      0.47      0.62        72
  interest_2       0.00      0.00      0.00         2
  interest_3       1.00      0.38      0.56        13
  interest_4       0.91      0.28      0.43        36
  interest_5       0.80      0.82      0.81       100
  interest_6       0.78      1.00      0.88       251

    accuracy                           0.80       474
   macro avg       0.74      0.49      0.55       474
weighted avg       0.82      0.80      0.78       474



In [17]:
# 9) 새 문장 예측 함수
def predict_sense(sentence: str) -> str:
    """입력 문장을 받아 학습된 분류기로 'interest'의 의미 레이블 예측."""
    return pipeline.predict([sentence])[0]

# 10) 데모 예측
demo_sentences = [
    "There is growing interest among investors in the new fund.",
    "He paid interest on the loan for ten years.",
    "Her main interest lies in classical music.",
]
print("🔎 [데모 문장 예측 결과]")
for s in demo_sentences:
    print(f"문장: {s}")
    print(f" → 예측된 의미(sense): {predict_sense(s)}")
    print("-"*50)

🔎 [데모 문장 예측 결과]
문장: There is growing interest among investors in the new fund.
 → 예측된 의미(sense): interest_1
--------------------------------------------------
문장: He paid interest on the loan for ten years.
 → 예측된 의미(sense): interest_6
--------------------------------------------------
문장: Her main interest lies in classical music.
 → 예측된 의미(sense): interest_6
--------------------------------------------------


In [18]:
# 11) 클래스별 중요 특징 단어 Top-5
clf = pipeline.named_steps["clf"]
tfidf = pipeline.named_steps["tfidf"]
feature_names = np.array(tfidf.get_feature_names_out())
classes = clf.classes_

print("💡 [각 의미를 구분하는 중요한 단어 Top 5]")
for idx, cls in enumerate(classes):
    top_idx = np.argsort(clf.coef_[idx])[::-1][:5]
    print(f"- {cls}: {', '.join(feature_names[top_idx])}")

💡 [각 의미를 구분하는 중요한 단어 Top 5]
- interest_1: interest in, buying, buying interest, investor interest, investor
- interest_2: might, baseball, earthquake, florida, of interest
- interest_3: pursue, to pursue, pursue other, interests, resigned
- interest_4: interests, best, interests of, not, public
- interest_5: interests, short interest, interests in, short, in
- interest_6: rates, interest rates, rate, bonds, interest rate
